# 🎥 Entrega 1 — OpenPose sobre o KIMORE no Google Colab

Extrai os **keypoints (BODY_25)** dos vídeos clínicos usando a **GPU gratuita do Colab**
(T4, 16 GB — bem mais forte que uma GPU de notebook). O OpenPose só gera os JSON; a
análise (ângulos, desvios, relatório) usa o nosso código do repositório, que roda igual
aqui ou na máquina local.

**Fluxo:** build do OpenPose → subir vídeos do KIMORE → rodar OpenPose → analisar → baixar.

> ⚠️ **Aviso honesto:** compilar o OpenPose no Colab é historicamente **chato** (o OpenPose
> 1.7 mira CUDA/cuDNN antigos). As células abaixo trazem a receita mais comum e os contornos
> conhecidos; se algum passo quebrar, é questão de ajustar flags — não desista no primeiro erro.

**Runtime → Alterar tipo de ambiente de execução → GPU** antes de começar.

## 0. Conferir a GPU

In [ ]:
!nvidia-smi

## 1. Build do OpenPose (v1.7.0)
Leva ~10–20 min. Compilamos **com CUDA e sem cuDNN** (`-DUSE_CUDNN=OFF`), que é o contorno
mais estável no Colab.

In [ ]:
# dependências de sistema
!apt-get -qq update
!apt-get -qq install -y libatlas-base-dev libprotobuf-dev libleveldb-dev \n    libsnappy-dev libhdf5-serial-dev protobuf-compiler libgflags-dev \n    libgoogle-glog-dev liblmdb-dev opencl-headers ocl-icd-opencl-dev libviennacl-dev > /dev/null


In [ ]:
import os
if not os.path.exists('openpose'):
    !git clone --depth 1 -b v1.7.0 https://github.com/CMU-Perceptual-Computing-Lab/openpose.git


In [ ]:
# Baixar os modelos (pose BODY_25). O getModels.sh aponta para um servidor da CMU
# que VIVE FORA DO AR. Se falhar, use o mirror comentado logo abaixo.
%cd /content/openpose/models
!chmod +x getModels.sh && ./getModels.sh || echo '⚠️ getModels falhou — use o mirror abaixo'

# --- MIRROR (descomente se o passo acima falhou) ---
# !pip -q install gdown
# import gdown; gdown.download('https://drive.google.com/uc?id=1QCSxJZpnWvM00hx49CJ2zky7PWGzpcEh',
#                              'pose/body_25/pose_iter_584000.caffemodel', quiet=False)
%cd /content/openpose

In [ ]:
# Compilar
!mkdir -p build
%cd build
!cmake .. -DUSE_CUDNN=OFF -DBUILD_PYTHON=OFF -DDOWNLOAD_BODY_25_MODEL=OFF > /tmp/cmake.log 2>&1 || tail -30 /tmp/cmake.log
!make -j`nproc` > /tmp/make.log 2>&1 || tail -40 /tmp/make.log
%cd /content/openpose
!ls build/examples/openpose/openpose.bin && echo '✅ build OK'

## 2. Subir os vídeos do KIMORE
Duas opções — escolha uma. O KIMORE é grande, então suba **só alguns vídeos** (uns poucos
sujeitos saudáveis + com disfunção já bastam para o demo).

In [ ]:
# Opção A: montar o Google Drive (recomendado p/ arquivos grandes)
from google.colab import drive
drive.mount('/content/drive')
# ajuste para a pasta onde você colocou os vídeos no seu Drive:
VIDEO_DIR = '/content/drive/MyDrive/kimore_videos'

# Opção B: upload manual (arquivos pequenos)
# from google.colab import files; up = files.upload()
# import os; VIDEO_DIR = '/content'; 

import glob
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4') + glob.glob(f'{VIDEO_DIR}/*.avi'))
print(len(videos), 'vídeos:', [os.path.basename(v) for v in videos])

## 3. Rodar o OpenPose → JSON por frame
Gera uma pasta de JSONs por vídeo em `/content/json/<nome>`. `--net_resolution` menor =
mais rápido / menos memória (a T4 do Colab aguenta a resolução padrão, mas mantemos econômico).

In [ ]:
import os, subprocess
BIN = '/content/openpose/build/examples/openpose/openpose.bin'
os.makedirs('/content/json', exist_ok=True)

for v in videos:
    name = os.path.splitext(os.path.basename(v))[0]
    out = f'/content/json/{name}'
    os.makedirs(out, exist_ok=True)
    print('▶', name)
    subprocess.run([BIN, '--video', v, '--write_json', out,
                    '--model_pose', 'BODY_25', '--net_resolution', '-1x256',
                    '--display', '0', '--render_pose', '0'],
                   cwd='/content/openpose', check=True)
    print('   frames:', len(os.listdir(out)))

## 4. Analisar com o nosso código (ângulos + desvios + relatório)
Clona o repositório do projeto e roda o pipeline `src.video.cli` sobre cada pasta de JSONs.

In [ ]:
%cd /content
# se o repo for privado, use um token; se público, a URL direta basta
![ -d tech-challenge ] || git clone <URL_DO_SEU_REPO_GIT> tech-challenge
%cd /content/tech-challenge
!pip -q install -r requirements.txt

In [ ]:
import glob, os
for jdir in sorted(glob.glob('/content/json/*')):
    !python -m src.video.cli --json-dir "{jdir}" --fps 30 --out reports
!ls -R reports

## 5. Baixar os resultados
Compacta os JSONs e os relatórios para você levar para a máquina local / anexar ao repo.

In [ ]:
!zip -qr /content/resultados_openpose.zip /content/json reports
from google.colab import files
files.download('/content/resultados_openpose.zip')